# tt-mlir #8570 repro check

런타임: CPU 고RAM 선택 → **런타임 → 모두 실행**만 하면 됩니다. 완료되면 GitHub에 저장(File → Save a copy in GitHub)해주세요.

In [ ]:
%cd /content
!rm -rf /content/tt-mlir
!git clone --branch main https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git remote add upstream https://github.com/tenstorrent/tt-mlir.git
!git fetch upstream main --quiet
!git reset --hard upstream/main
!git log --oneline -1

In [ ]:
# toolchain 빌드 (최소 구성: RUNTIME/PythonBindings/OPMODEL 전부 OFF, StableHLO는 ON 필요)
!apt-get -qq update && apt-get -qq install -y ninja-build clang lld ccache > /tmp/apt.log 2>&1
!cmake -B /content/tt-mlir/env/build /content/tt-mlir/env \
    -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++ -G Ninja
!cmake --build /content/tt-mlir/env/build 2>&1 | tail -60

In [ ]:
import os
os.environ["TTMLIR_TOOLCHAIN_DIR"] = "/opt/ttmlir-toolchain"

!cmake -B /content/tt-mlir/build /content/tt-mlir \
    -G Ninja \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++ \
    -DTTMLIR_ENABLE_RUNTIME=OFF \
    -DTTMLIR_ENABLE_RUNTIME_TESTS=OFF \
    -DTTMLIR_ENABLE_PYKERNEL=OFF \
    -DTT_RUNTIME_ENABLE_PERF_TRACE=OFF \
    -DTTMLIR_ENABLE_OPMODEL=OFF \
    -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF 2>&1 | tail -40

In [ ]:
!ninja -C /content/tt-mlir/build ttmlir-opt 2>&1 | tail -80

In [ ]:
import subprocess
r = subprocess.run(["/content/tt-mlir/build/bin/ttmlir-opt", "--version"], capture_output=True, text=True, timeout=30)
print("rc:", r.returncode)
print(r.stdout)
print(r.stderr)

In [ ]:
repro_mlir = r"""// SPDX-FileCopyrightText: (c) 2026 Tenstorrent AI ULC
//
// SPDX-License-Identifier: Apache-2.0

// REQUIRES: stablehlo
// RUN: ttmlir-opt --convert-stablehlo-to-ttir %s | FileCheck %s

// Repro for https://github.com/tenstorrent/tt-mlir/issues/8570
// scatter with leading update_window_dims=[0] (Qwen 3.5 27B MRoPE position_ids pattern)

module @SyncTensorsGraph.45 attributes {mhlo.is_dynamic = false} {
  func.func @main(%arg0: tensor<3x1x494xi64>,
                  %arg1: tensor<3x494xi64>,
                  %arg2: tensor<494xi64>) -> tensor<3x1x494xi64> {
    %c   = stablehlo.constant dense<494> : tensor<494xi64>
    %c_0 = stablehlo.constant dense<0>   : tensor<494xi64>
    %0 = stablehlo.reshape %arg0 : (tensor<3x1x494xi64>) -> tensor<3x494xi64>
    %1 = stablehlo.reshape %arg2 : (tensor<494xi64>)     -> tensor<1x1x494xi64>
    %2 = stablehlo.reshape %1    : (tensor<1x1x494xi64>) -> tensor<494xi64>
    %3 = stablehlo.compare LT, %2, %c_0 : (tensor<494xi64>, tensor<494xi64>) -> tensor<494xi1>
    %4 = stablehlo.add %2, %c    : tensor<494xi64>
    %5 = stablehlo.select %3, %4, %2 : tensor<494xi1>, tensor<494xi64>
    %6 = stablehlo.reshape %5    : (tensor<494xi64>)     -> tensor<494x1xi64>
    %7 = stablehlo.reshape %arg1 : (tensor<3x494xi64>)   -> tensor<1x3x494xi64>
    %8 = stablehlo.reshape %7    : (tensor<1x3x494xi64>) -> tensor<3x494xi64>
    %9 = "stablehlo.scatter"(%0, %6, %8) <{
           scatter_dimension_numbers = #stablehlo.scatter<
             update_window_dims = [0], inserted_window_dims = [1],
             scatter_dims_to_operand_dims = [1], index_vector_dim = 1>
         }> ({
      ^bb0(%a: tensor<i64>, %b: tensor<i64>):
        stablehlo.return %b : tensor<i64>
      }) : (tensor<3x494xi64>, tensor<494x1xi64>, tensor<3x494xi64>) -> tensor<3x494xi64>
    %10 = stablehlo.reshape %9 : (tensor<3x494xi64>) -> tensor<3x1x494xi64>
    return %10 : tensor<3x1x494xi64>
  }
}
"""
with open("/content/repro_8570.mlir", "w") as f:
    f.write(repro_mlir)
print("written")

In [ ]:
import subprocess
r = subprocess.run(
    ["/content/tt-mlir/build/bin/ttmlir-opt", "--convert-stablehlo-to-ttir", "/content/repro_8570.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== rc:", r.returncode, "===")
print("=== stdout ===")
print(r.stdout)
print("=== stderr ===")
print(r.stderr)

## 결과 판독

- **rc != 0 + stderr에 `ttir.repeat` verifier 에러** → 버그 아직 살아있음, #8570 재현 확인, #8594 여전히 유효.
- **rc == 0** → 이미 고쳐져 있음, `ttir.scatter` 결과 IR이 stdout에 찍혀야 함. #8570은 사실상 해결된 상태.
- 이 셀 실행 후 **File → Save a copy in GitHub**로 저장 부탁드립니다 (브랜치: 이 노트북이 push된 브랜치 그대로).